In [45]:
import pandas as pd
import eurostat
import time
import numpy as np
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

In [ ]:
year = 2022

def fetch_eu_air_traffic():
	print("1. Fetching Eurostat Air Traffic Data...")
	# Dataset 'avia_paocc' = Passenger transport by partner country
	# Note: Eurostat datasets are massive. We download it and filter it.
	dataset_code = 'avia_paocc'
	
	try:
		df = eurostat.get_data_df(dataset_code)
		year_col = str(year)
		if year_col in df.columns:
			# Filter columns: reporting country (geo), partner country (partner), and traffic
			df_traffic = df[['geo\\TIME_PERIOD', 'partner', year_col]].dropna()
			df_traffic.rename(columns={'geo\\TIME_PERIOD': 'Origin', 'partner': 'Destination', year_col: 'Passengers'}, inplace=True)
			print(f"   -> Successfully found {len(df_traffic)} traffic records.")
			
			# 1. Drop rows where traffic is zero or missing (NaN)
			df_clean = df_traffic[df_traffic['Passengers'] > 0]

			# 2. Filter out geopolitical aggregates (EU27_2020, EA19, etc.)
			# Eurostat country codes are exactly 2 letters long (e.g., 'SI', 'FR', 'DE').
			df_clean = df_clean[
				(df_clean['Origin'].str.len() == 2) & 
				(df_clean['Destination'].str.len() == 2)
			]

			# 3. Optional: Remove domestic flights (where Origin and Destination are the same country)
			# If your model only cares about international spread
			df_clean = df_clean[df_clean['Origin'] != df_clean['Destination']]

			print(f"   -> After cleanup succesfully loaded {len(df_traffic)} traffic records.")
			return df_clean
		else:
			print("   -> Year not found in dataset. Please check Eurostat column names.")
			return None
	except Exception as e:
		print(f"Failed to fetch Eurostat data: {e}")
		return None

In [21]:
df_traffic = fetch_eu_air_traffic()

1. Fetching Eurostat Air Traffic Data...
   -> Successfully loaded 31058 traffic records.
Failed to fetch Eurostat data: 'Origin'


In [59]:
dataset_code = 'rail_pa_typepas'
df = eurostat.get_data_df(dataset_code)
year_col = str(2022)
if year_col in df.columns:
    # Filter columns: reporting country (geo), partner country (partner), and traffic
    df_traffic = df[['geo\\TIME_PERIOD', 'partner', year_col]].dropna()
    df_traffic.rename(columns={'geo\\TIME_PERIOD': 'Origin', 'partner': 'Destination', year_col: 'Passengers'}, inplace=True)
    print(f"   -> Successfully found {len(df_traffic)} traffic records.")
    
    # 1. Drop rows where traffic is zero or missing (NaN)
    df_clean = df_traffic[df_traffic['Passengers'] > 0]

    # 2. Filter out geopolitical aggregates (EU27_2020, EA19, etc.)
    # Eurostat country codes are exactly 2 letters long (e.g., 'SI', 'FR', 'DE').
    df_clean = df_clean[
        (df_clean['Origin'].str.len() == 2) & 
        (df_clean['Destination'].str.len() == 2)
    ]

    # 3. Optional: Remove domestic flights (where Origin and Destination are the same country)
    # If your model only cares about international spread
    df_clean = df_clean[df_clean['Origin'] != df_clean['Destination']]

    print(f"   -> After cleanup succesfully loaded {len(df_traffic)} traffic records.")

KeyError: "['partner'] not in index"

In [60]:
df.head()

,freq,unit,tra_cov,geo\TIME_PERIOD,2004,2005,2006,2007,2008,2009,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,MIO_PKM,INTL,AT,1105.0,1147.0,1211.0,1279.0,1452.0,1442.0,...,424.0,423.0,456.0,492.0,190.0,232.0,598.0,698.0,665.0,NaN
1,A,MIO_PKM,INTL,BE,550.0,740.0,774.0,856.0,1226.0,1232.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A,MIO_PKM,INTL,BG,NaN,NaN,45.0,62.0,52.0,49.0,...,14.0,18.0,18.0,24.0,6.0,4.0,32.0,36.0,30.0,32.0
3,A,MIO_PKM,INTL,CH,NaN,NaN,NaN,NaN,912.0,879.0,...,1037.0,1053.0,1088.0,1189.0,487.0,565.0,1148.0,1318.0,1249.0,NaN
4,A,MIO_PKM,INTL,CZ,368.0,381.0,358.0,362.0,449.0,340.0,...,1229.0,1437.0,1705.0,1884.0,538.0,603.0,1555.0,1716.0,1839.0,NaN


In [29]:
df_clean.sort_values("Passengers").head()

,Origin,Destination,Passengers
11926,BA,HU,1.0
11921,SI,HR,1.0
6446,SI,IT,1.0
6489,BA,LU,1.0
6412,BA,IT,1.0


In [ ]:
df_clean.tail

In [38]:
# Use pivot_table to create the matrix
# Origin becomes the Y-axis (index), Destination becomes the X-axis (columns)
traffic_matrix = pd.pivot_table(
    df_clean, 
    values='Passengers', 
    index='Origin', 
    columns='Destination', 
    aggfunc='sum',     # Sums the passengers if there are duplicate route entries
)

# Optional but recommended: Ensure the matrix is perfectly square.
# Sometimes a country only receives flights but doesn't send them (or vice versa in the data).
# This aligns the rows and columns so they match exactly.
all_countries = sorted(list(set(traffic_matrix.index).union(set(traffic_matrix.columns))))
traffic_matrix = traffic_matrix.reindex(index=all_countries, columns=all_countries)

imputed_matrix = traffic_matrix.combine_first(traffic_matrix.T)
final_matrix = imputed_matrix.fillna(0)
# Convert to a pure NumPy array for your SIR model math
numpy_matrix = final_matrix.to_numpy()



In [39]:
final_matrix

,AT,BA,BE,BG,CH,CY,CZ,DE,DK,EE,...,NL,NO,PL,PT,RO,RS,SE,SI,SK,TR
Origin,,,,,,,,,,,,,,,,,,,,,
AT,0.0,1305220.0,4331620.0,3558148.0,6230384.0,2700316.0,1252656.0,33571556.0,3350228.0,415408.0,...,7669008.0,1298784.0,4130868.0,2827356.0,4965416.0,2304400.0,3311112.0,804.0,869616.0,11074276.0
BA,1305220.0,0.0,102328.0,3752.0,205688.0,0.0,20.0,3949584.0,397040.0,0.0,...,341452.0,129648.0,101416.0,0.0,412.0,240760.0,971724.0,0.0,0.0,3299476.0
BE,4531180.0,102328.0,0.0,1495696.0,5374336.0,429996.0,2040276.0,9885588.0,2980272.0,294676.0,...,1475264.0,1777148.0,3885492.0,11270172.0,4957764.0,114992.0,3235516.0,459252.0,313628.0,8237584.0
BG,3543824.0,3752.0,1464812.0,0.0,535076.0,1060944.0,2453892.0,13081640.0,952188.0,113604.0,...,2678832.0,402008.0,4923564.0,235520.0,313800.0,155012.0,158652.0,0.0,832716.0,3365496.0
CH,6343632.0,205688.0,5337176.0,816160.0,0.0,1162608.0,2552796.0,28270652.0,5046308.0,322216.0,...,11532788.0,1820396.0,3956004.0,20842252.0,2197280.0,3566732.0,4091780.0,461016.0,7712.0,18173872.0
CY,2699444.0,0.0,447108.0,1065640.0,1177424.0,0.0,530696.0,4330880.0,1314884.0,93456.0,...,927872.0,705172.0,3769572.0,96.0,1437608.0,767556.0,1734776.0,140.0,636804.0,12.0
CZ,1261740.0,20.0,2040488.0,2504336.0,2163980.0,530800.0,0.0,5040444.0,1567192.0,808.0,...,4069312.0,876252.0,2000920.0,1410468.0,659792.0,217460.0,1048868.0,2848.0,300980.0,6120052.0
DE,33412324.0,3949584.0,9818592.0,13487084.0,28113620.0,4273212.0,5006832.0,0.0,15218612.0,2910120.0,...,22444024.0,11608568.0,24103800.0,35469052.0,17993856.0,7177260.0,16745676.0,1189188.0,37108.0,139539324.0
DK,3350264.0,397040.0,2981864.0,943052.0,4401804.0,1316440.0,1565924.0,12416544.0,0.0,541500.0,...,12084700.0,16643192.0,6654832.0,3123372.0,1915344.0,373700.0,9053892.0,496.0,259604.0,9639908.0


In [74]:
import pandas as pd
import eurostat

print("Fetching Eurostat Population Data...")

# 'demo_pjan' is the official dataset for population on Jan 1st
dataset_code = 'demo_pjan'

# Download the raw dataset
df_pop = eurostat.get_data_df(dataset_code)

# Eurostat demographic data is highly segmented. 
# We only want: Total Sex ('T') and Total Age ('TOTAL')
if 'sex' in df_pop.columns and 'age' in df_pop.columns:
    df_pop = df_pop[(df_pop['sex'] == 'T') & (df_pop['age'] == 'TOTAL')]

# Check if our target year exists in the dataset
if year_col in df_pop.columns:
    # Keep only the country code (geo\time) and the population for that year
    df_clean_pop = df_pop[['geo\\TIME_PERIOD', year_col]].copy()
    df_clean_pop.rename(columns={'geo\\TIME_PERIOD': 'Country_Code', year_col: 'Population'}, inplace=True)
    
    # Filter out the non-country aggregates (e.g., 'EU27_2020', 'EA19')
    # Country codes are exactly 2 characters long
    df_clean_pop = df_clean_pop[df_clean_pop['Country_Code'].str.len() == 2]
    
    print(f"   -> Successfully loaded populations for {len(df_clean_pop)} entities.")
    # You can convert this into a quick dictionary for your SIR model:
    if df_clean_pop is not None:
        # Creates a dictionary like: {'FR': 67871928.0, 'DE': 83237124.0}
        pop_dict = df_clean_pop['Country_Code','Population']

            

Fetching Eurostat Population Data...
   -> Successfully loaded populations for 50 entities.


KeyError: ('Country_Code', 'Population')

In [75]:
df_clean_pop

,Country_Code,Population
118,AD,NaN
119,AL,2793592.0
120,AM,NaN
121,AT,8978929.0
122,AZ,10156366.0
123,BA,NaN
124,BE,11617623.0
125,BG,6482484.0
126,BY,NaN
127,CH,8738791.0


In [55]:
populations = np.zeros(len(final_matrix.columns.to_list()))
for i, col in enumerate(final_matrix.columns.to_list()):
    populations[i] = pop_dict.get(col)

In [67]:
pop_dict

{'AD': nan,
 'AL': 2793592.0,
 'AM': nan,
 'AT': 8978929.0,
 'AZ': 10156366.0,
 'BA': nan,
 'BE': 11617623.0,
 'BG': 6482484.0,
 'BY': nan,
 'CH': 8738791.0,
 'CY': 929764.0,
 'CZ': 10516707.0,
 'DE': 83237124.0,
 'DK': 5873420.0,
 'EE': 1331796.0,
 'EL': 10461627.0,
 'ES': 47486843.0,
 'FI': 5548241.0,
 'FR': 68091703.0,
 'FX': nan,
 'GE': 3688647.0,
 'HR': 3862305.0,
 'HU': 9610403.0,
 'IE': 5154277.0,
 'IS': 376248.0,
 'IT': 59030133.0,
 'LI': 39308.0,
 'LT': 2805998.0,
 'LU': 645397.0,
 'LV': 1875757.0,
 'MC': nan,
 'MD': 2565030.0,
 'ME': 617683.0,
 'MK': 1837114.0,
 'MT': 520174.0,
 'NL': 17590672.0,
 'NO': 5425270.0,
 'PL': 36889761.0,
 'PT': 10421117.0,
 'RO': 19043098.0,
 'RS': 6797105.0,
 'RU': nan,
 'SE': 10452326.0,
 'SI': 2107180.0,
 'SK': 5434712.0,
 'SM': nan,
 'TR': 84680273.0,
 'UA': 40997698.0,
 'UK': nan,
 'XK': 1773971.0}

In [65]:
final_matrix.columns.to_list()

['AT',
 'BA',
 'BE',
 'BG',
 'CH',
 'CY',
 'CZ',
 'DE',
 'DK',
 'EE',
 'EL',
 'ES',
 'FI',
 'FR',
 'HR',
 'HU',
 'IE',
 'IS',
 'IT',
 'LT',
 'LU',
 'LV',
 'ME',
 'MK',
 'MT',
 'NL',
 'NO',
 'PL',
 'PT',
 'RO',
 'RS',
 'SE',
 'SI',
 'SK',
 'TR']

In [56]:
populations

array([ 8978929.,       nan, 11617623.,  6482484.,  8738791.,   929764.,
       10516707., 83237124.,  5873420.,  1331796., 10461627., 47486843.,
        5548241., 68091703.,  3862305.,  9610403.,  5154277.,   376248.,
       59030133.,  2805998.,   645397.,  1875757.,   617683.,  1837114.,
         520174., 17590672.,  5425270., 36889761., 10421117., 19043098.,
        6797105., 10452326.,  2107180.,  5434712., 84680273.])

In [66]:
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
import imageio
import os

# ==========================================
# 1. SETUP GEOMETRY & MOCK DATA

print("Downloading map geometry from Natural Earth...")
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

# Natural Earth capitalizes their original columns, so we rename them 
# to lowercase so the rest of our script works perfectly.
world.rename(columns={'CONTINENT': 'continent', 'NAME': 'name', 'ISO_A3': 'iso_a3'}, inplace=True)
# Filter to Europe (excluding Russia for a cleaner zoom)
# THE FIX: Manually patch France and Norway before filtering
world.loc[world['name'] == 'France', 'iso_a3'] = 'FRA'
world.loc[world['name'] == 'Norway', 'iso_a3'] = 'NOR'
europe = world[(world.continent == 'Europe') & (world.name != 'Russia')].copy()

# Ensure we have ISO3 codes to match your model
europe = europe[europe.iso_a3 != '-99']

time_steps = 30
countries = europe['iso_a3'].tolist()
num_nodes = len(countries)

# Generate mock SIR data (Shape: [Days, Countries, 3])
np.random.seed(42)
sir_data = np.random.rand(time_steps, num_nodes, 3)
sir_data = sir_data / sir_data.sum(axis=2, keepdims=True) # Normalize S+I+R = 1

# ==========================================
# 2. RENDER FRAMES TO TEMP FOLDER
# ==========================================
print("Generating frames...")
filenames = []

# Create a temporary folder for the PNG frames
if not os.path.exists('temp_frames'):
    os.makedirs('temp_frames')

for t in range(time_steps):
    # Setup the 2x2 grid
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f'EU Disease Spread - Day {t}', fontsize=16)
    
    # Extract data for this specific day
    day_data = sir_data[t]
    
    # Add the colors to our GeoDataFrame
    # Matplotlib expects RGB colors as tuples from (0.0 to 1.0)
    europe['Color_S'] = [ (0, s, 0) for s in day_data[:, 0] ]       # Pure Green
    europe['Color_I'] = [ (i, 0, 0) for i in day_data[:, 1] ]       # Pure Red
    europe['Color_R'] = [ (0, 0, r) for r in day_data[:, 2] ]       # Pure Blue
    europe['Color_Comb'] = [ (i, s, r) for i, s, r in day_data ]    # RGB Combined
    
    # Define mapping for the 4 subplots
    plot_configs = [
        (axs[0, 0], 'Color_S', 'Susceptible (Green)'),
        (axs[0, 1], 'Color_I', 'Infected (Red)'),
        (axs[1, 0], 'Color_R', 'Recovered (Blue)'),
        (axs[1, 1], 'Color_Comb', 'Combined (RGB)')
    ]
    
    # Draw the maps
    for ax, color_col, title in plot_configs:
        europe.plot(ax=ax, color=europe[color_col], edgecolor='black', linewidth=0.5)
        ax.set_title(title)
        
        # Crop the view nicely around Europe
        ax.set_xlim(-25, 45)
        ax.set_ylim(35, 75)
        ax.axis('off') # Hide lat/lon axes for a cleaner look
    
    # Save the frame
    filename = f'temp_frames/frame_{t:03d}.png'
    plt.tight_layout()
    plt.savefig(filename, dpi=100, bbox_inches='tight', facecolor='white')
    plt.close(fig) # Free up memory
    filenames.append(filename)
    
    print(f"Rendered Day {t}/{time_steps-1}")

# ==========================================
# 3. STITCH INTO GIF
# ==========================================
print("Stitching into GIF...")
gif_path = 'sir_eu_model.gif'

# Read PNGs and append to a GIF (fps controls the animation speed)
with imageio.get_writer(gif_path, mode='I', fps=4) as writer:
    for filename in filenames:
        image = imageio.imread(filename)
        writer.append_data(image)

# Optional: Cleanup temporary PNG files
for filename in filenames:
    os.remove(filename)
os.rmdir('temp_frames')

print(f"Success! GIF saved as: {gif_path}")

Generating frames...
Rendered Day 0/29
Rendered Day 1/29
Rendered Day 2/29
Rendered Day 3/29
Rendered Day 4/29
Rendered Day 5/29
Rendered Day 6/29
Rendered Day 7/29
Rendered Day 8/29
Rendered Day 9/29
Rendered Day 10/29
Rendered Day 11/29
Rendered Day 12/29
Rendered Day 13/29
Rendered Day 14/29
Rendered Day 15/29
Rendered Day 16/29
Rendered Day 17/29
Rendered Day 18/29
Rendered Day 19/29
Rendered Day 20/29
Rendered Day 21/29
Rendered Day 22/29
Rendered Day 23/29
Rendered Day 24/29
Rendered Day 25/29
Rendered Day 26/29
Rendered Day 27/29
Rendered Day 28/29
Rendered Day 29/29
Stitching into GIF...


C:\Users\pilis\AppData\Local\Temp\ipykernel_15412\3528970824.py:96: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  image = imageio.imread(filename)


Success! GIF saved as: sir_eu_model.gif


In [62]:

import sys
import subprocess

# This forces installation into the active Python environment
subprocess.check_call([sys.executable, "-m", "pip", "install", "geopandas"])

# Now try importing it
import geopandas
print("geopandas successfully imported!")

geopandas successfully imported!
